# 💥 Notebook 5: Failures & Heartbeats

How Temporal survives crashes and handles long-running activities.

## Learning Objectives

By the end of this notebook, you'll understand:
- How Temporal detects worker failures
- What heartbeats are and why they matter
- How workflow replay works
- Activity timeouts and their purposes

In [ ]:
import asyncio
from datetime import timedelta
from temporalio import activity, workflow
from temporalio.client import Client
from temporalio.common import RetryPolicy
from temporalio.worker import Worker
import uuid
import time

print("✅ Temporal SDK imported!")

## 💥 The Crash Problem Revisited

In [ ]:
print("💥 What Happens When Workers Crash?")
print("=" * 60)
print("""
WITHOUT TEMPORAL:
─────────────────────────────────────────────────────────────
t=0:  Start processing order
t=1:  Charge payment ✅
t=2:  Reserve inventory ✅
t=3:  CRASH! 💥 Container dies
t=∞:  Shipping never created
      Customer charged but no delivery!

WITH TEMPORAL:
─────────────────────────────────────────────────────────────
t=0:  Start workflow
t=1:  Activity: charge_payment ──► saved to history
t=2:  Activity: reserve_inventory ──► saved to history
t=3:  CRASH! 💥 Worker dies
      
      ... Temporal Server notices worker is gone ...
      
t=10: Another worker picks up the workflow
t=10: REPLAY: charge_payment ──► skip (in history)
t=10: REPLAY: reserve_inventory ──► skip (in history)
t=11: Activity: create_shipping ──► EXECUTES!
t=12: Workflow completes ✅

KEY INSIGHT:
Activities that COMPLETED are NOT re-run.
Their results are loaded from history!
""")

## ⏰ Activity Timeouts

In [ ]:
print("⏰ Understanding Activity Timeouts")
print("=" * 60)
print("""
Temporal uses MULTIPLE timeouts for different scenarios:

1. START_TO_CLOSE_TIMEOUT
   ─────────────────────────────────────────────────────────
   Max time for a SINGLE attempt of an activity.
   
   Activity starts ──────────────────► Must complete
                    [start_to_close]
   
   If exceeded: Activity attempt fails, may retry.

2. SCHEDULE_TO_CLOSE_TIMEOUT
   ─────────────────────────────────────────────────────────
   Max time from scheduling to completion (ALL retries).
   
   Scheduled ─────────────────────────────► Must complete
             [schedule_to_close]
   
   If exceeded: Activity permanently fails.

3. HEARTBEAT_TIMEOUT
   ─────────────────────────────────────────────────────────
   Max time between heartbeats for long activities.
   
   Activity ──💓──💓──💓──💓──💓──► Complete
            [heartbeat interval must be < timeout]
   
   If exceeded: Worker assumed dead, activity retried!
""")

## 💓 Heartbeats Explained

In [ ]:
print("💓 Why Heartbeats Matter")
print("=" * 60)
print("""
SCENARIO: Activity that takes 10 minutes

WITHOUT HEARTBEAT:
─────────────────────────────────────────────────────────────
  start_to_close_timeout = 15 minutes
  
  t=0:   Activity starts on Worker A
  t=5:   Worker A CRASHES! 💥
  t=15:  Timeout finally fires
  t=15:  Temporal retries on Worker B
  
  10 MINUTES WASTED waiting for timeout!

WITH HEARTBEAT:
─────────────────────────────────────────────────────────────
  heartbeat_timeout = 30 seconds
  Activity sends heartbeat every 10 seconds
  
  t=0:   Activity starts on Worker A, 💓
  t=10:  💓
  t=20:  💓
  t=30:  Worker A CRASHES! 💥
  t=30:  No heartbeat received!
  t=60:  Temporal detects failure (30s timeout)
  t=60:  Activity retried on Worker B
  
  Only 30 seconds lost!

HEARTBEATS ARE ESSENTIAL FOR:
• Long-running activities (minutes to hours)
• Activities that process large datasets
• Activities with progress you want to preserve
""")

In [ ]:
from dataclasses import dataclass

@dataclass
class ProcessingInput:
    job_id: str
    total_items: int

@dataclass 
class ProcessingResult:
    job_id: str
    items_processed: int

@activity.defn
async def process_large_batch(input: ProcessingInput) -> ProcessingResult:
    print(f"\n   🔄 Processing {input.total_items} items...")
    
    for i in range(input.total_items):
        await asyncio.sleep(0.5)
        
        activity.heartbeat(f"Processed {i + 1}/{input.total_items}")
        print(f"      💓 Heartbeat: {i + 1}/{input.total_items}")
    
    return ProcessingResult(
        job_id=input.job_id,
        items_processed=input.total_items
    )

@workflow.defn
class BatchProcessingWorkflow:
    @workflow.run
    async def run(self, input: ProcessingInput) -> ProcessingResult:
        print(f"\n🚀 Starting batch processing workflow")
        
        result = await workflow.execute_activity(
            process_large_batch,
            input,
            start_to_close_timeout=timedelta(minutes=5),
            heartbeat_timeout=timedelta(seconds=10),
        )
        
        print(f"   ✅ Processed {result.items_processed} items!")
        return result

print("✅ Heartbeat activity and workflow defined!")

In [ ]:
async def run_heartbeat_demo():
    client = await Client.connect("localhost:7233")
    print("✅ Connected to Temporal!")
    
    async with Worker(
        client,
        task_queue="batch-queue",
        workflows=[BatchProcessingWorkflow],
        activities=[process_large_batch]
    ):
        print("\n👷 Worker started!")
        
        input_data = ProcessingInput(
            job_id=str(uuid.uuid4())[:8],
            total_items=5
        )
        
        result = await client.execute_workflow(
            BatchProcessingWorkflow.run,
            input_data,
            id=f"batch-{input_data.job_id}",
            task_queue="batch-queue"
        )
        
        return result

print("💓 Heartbeat Demo")
print("=" * 60)

try:
    result = await run_heartbeat_demo()
    print(f"\n✅ Workflow completed! Processed {result.items_processed} items.")
except Exception as e:
    print(f"\n❌ Error: {e}")
    print("   Make sure: docker compose up -d")

## 🔄 Workflow Replay Deep Dive

In [ ]:
print("🔄 How Workflow Replay Works")
print("=" * 60)
print("""
When a workflow needs to continue (after crash or signal):

STEP 1: Load history from Temporal server
─────────────────────────────────────────────────────────────
  History for workflow-123:
  [1] WorkflowExecutionStarted
  [2] ActivityTaskScheduled (charge_payment)
  [3] ActivityTaskStarted
  [4] ActivityTaskCompleted (result: {txn: 'abc'})
  [5] ActivityTaskScheduled (reserve_inventory)
  [6] ActivityTaskStarted
  [7] ... crash happened here ...

STEP 2: Replay workflow code
─────────────────────────────────────────────────────────────
  Workflow code runs again from the beginning!
  
  But when it hits:
    await workflow.execute_activity(charge_payment, ...)
  
  Temporal says: "I have the result in history!"
  Returns {txn: 'abc'} WITHOUT calling the activity.

STEP 3: Continue from where we left off
─────────────────────────────────────────────────────────────
  When replay catches up to the crash point,
  workflow continues executing normally.
  
  reserve_inventory: Re-scheduled (was in progress)
  create_shipping: Will execute normally

THIS IS WHY WORKFLOWS MUST BE DETERMINISTIC!
─────────────────────────────────────────────────────────────
  If workflow code made different decisions on replay,
  it wouldn't match the history and would fail.
  
  ❌ Don't use: random(), time.now(), external state
  ✅ Use: workflow.random(), workflow.now(), activities
""")

## 🧪 Simulating a Crash

In [ ]:
crash_counter = {"count": 0}

@dataclass
class CrashTestInput:
    test_id: str

@activity.defn
async def step_one(input: CrashTestInput) -> str:
    print(f"   1️⃣ Step one executing...")
    await asyncio.sleep(0.3)
    return f"step1_result_{input.test_id}"

@activity.defn
async def step_two_crashes_once(input: CrashTestInput) -> str:
    print(f"   2️⃣ Step two executing...")
    await asyncio.sleep(0.3)
    
    crash_counter["count"] += 1
    if crash_counter["count"] == 1:
        print(f"      💥 SIMULATING CRASH!")
        raise Exception("Worker crashed!")
    
    return f"step2_result_{input.test_id}"

@activity.defn
async def step_three(input: CrashTestInput) -> str:
    print(f"   3️⃣ Step three executing...")
    await asyncio.sleep(0.3)
    return f"step3_result_{input.test_id}"

@workflow.defn
class CrashTestWorkflow:
    @workflow.run
    async def run(self, input: CrashTestInput) -> dict:
        print(f"\n🚀 CrashTest workflow starting...")
        
        result1 = await workflow.execute_activity(
            step_one, input,
            start_to_close_timeout=timedelta(seconds=30),
        )
        print(f"      ✅ Step 1 complete: {result1}")
        
        result2 = await workflow.execute_activity(
            step_two_crashes_once, input,
            start_to_close_timeout=timedelta(seconds=30),
            retry_policy=RetryPolicy(
                maximum_attempts=3,
                initial_interval=timedelta(seconds=1)
            )
        )
        print(f"      ✅ Step 2 complete: {result2}")
        
        result3 = await workflow.execute_activity(
            step_three, input,
            start_to_close_timeout=timedelta(seconds=30),
        )
        print(f"      ✅ Step 3 complete: {result3}")
        
        return {"step1": result1, "step2": result2, "step3": result3}

print("✅ Crash test workflow defined!")

In [ ]:
async def run_crash_test():
    crash_counter["count"] = 0
    
    client = await Client.connect("localhost:7233")
    print("✅ Connected to Temporal!")
    
    async with Worker(
        client,
        task_queue="crash-test-queue",
        workflows=[CrashTestWorkflow],
        activities=[step_one, step_two_crashes_once, step_three]
    ):
        print("\n👷 Worker started!")
        
        test_input = CrashTestInput(test_id=str(uuid.uuid4())[:8])
        
        result = await client.execute_workflow(
            CrashTestWorkflow.run,
            test_input,
            id=f"crash-test-{test_input.test_id}",
            task_queue="crash-test-queue"
        )
        
        return result

print("💥 Crash Recovery Demo")
print("=" * 60)
print("Step 2 will 'crash' on first attempt, then succeed on retry.")
print("Watch how Temporal handles it!\n")

try:
    result = await run_crash_test()
    print(f"\n📊 Final Result:")
    for k, v in result.items():
        print(f"   {k}: {v}")
    print("\n✅ Workflow recovered from crash!")
except Exception as e:
    print(f"\n❌ Error: {e}")

## 📊 What to See in Temporal UI

In [ ]:
print("📊 What to Look for in Temporal UI")
print("=" * 60)
print("""
Open http://localhost:8080 and find your workflow.

In the Event History, you'll see:

┌─────────────────────────────────────────────────────────────┐
│ Event History for crash-test-xxx                            │
├─────────────────────────────────────────────────────────────┤
│ 1  WorkflowExecutionStarted                                 │
│ 2  ActivityTaskScheduled (step_one)                         │
│ 3  ActivityTaskStarted                                      │
│ 4  ActivityTaskCompleted ✅                                 │
│ 5  ActivityTaskScheduled (step_two_crashes_once)            │
│ 6  ActivityTaskStarted                                      │
│ 7  ActivityTaskFailed ❌ "Worker crashed!"                  │
│ 8  ActivityTaskScheduled (step_two_crashes_once) ← RETRY    │
│ 9  ActivityTaskStarted                                      │
│ 10 ActivityTaskCompleted ✅                                 │
│ 11 ActivityTaskScheduled (step_three)                       │
│ 12 ActivityTaskStarted                                      │
│ 13 ActivityTaskCompleted ✅                                 │
│ 14 WorkflowExecutionCompleted ✅                            │
└─────────────────────────────────────────────────────────────┘

NOTICE:
• Step 2 was scheduled twice (events 5 and 8)
• First attempt failed (event 7)
• Second attempt succeeded (event 10)
• Steps 1 and 3 only ran once

This is the FULL audit trail!
""")

## 🧪 Quick Quiz

1. **What's the purpose of heartbeats?**

2. **Why doesn't Temporal re-run completed activities on replay?**

3. **What happens if heartbeat_timeout is exceeded?**

In [ ]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Purpose of heartbeats:")
print("   - Detect worker failures quickly")
print("   - Don't wait for long timeout")
print("   - Can include progress info")
print()
print("2. Why skip completed activities:")
print("   - Results are in history")
print("   - Avoids double-charging, etc.")
print("   - Makes replay fast")
print()
print("3. When heartbeat_timeout exceeded:")
print("   - Worker assumed dead")
print("   - Activity is retried")
print("   - On different worker if available")

## 📚 Summary

### Failure Detection

| Mechanism | Purpose | When to Use |
|-----------|---------|-------------|
| start_to_close_timeout | Single attempt limit | Always |
| heartbeat_timeout | Detect dead workers | Long activities |
| schedule_to_close_timeout | Total retry limit | Critical paths |

### Key Insights

1. **Heartbeats** = "I'm still alive" signals
2. **Replay** = Re-run workflow, skip completed activities
3. **History** = Source of truth for state
4. **Determinism** = Required for correct replay

### Next Up

In **Notebook 6**, we'll cover signals and timers:
- Waiting for external events
- Human-in-the-loop workflows
- Durable timers that survive crashes